# NB04 — Validatie en Consistentie

**Input:**
- `data/processed/comparisons.csv` — paarsgewijze vergelijkingen
- `data/processed/bt_params.csv` — geschatte BT log-sterkte parameters
- `data/raw/database.db` — voor responstijd-data uit de Timestamp-tabel

**Output:** `data/processed/respondent_quality.csv` — diagnostische metrics per respondent

**Doel:** Beoordeel de modelfit en datakwaliteit via vier indicatoren:
1. **Goodness-of-fit (deviance)** — hoe goed verklaart het BT-model de waargenomen vergelijkingen?
2. **Circulaire triaden per respondent** — telt intransitieve keuzes (A>B, B>C maar C>A)
3. **Kendall's W** — inter-beoordelaar overeenstemming over de impliciete rangordes
4. **Responstijd-analyse** — markeer verdacht snelle antwoorden

## Stap 1 — Data laden

Laad de vergelijkingen en de BT-parameters. We bouwen een opzoekdictionary
`intersection_id -> log-sterkte` om snel de parameterwaardes te kunnen opvragen
bij het berekenen van de log-likelihood.

In [2]:
import sqlite3
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import rankdata

BASE_DIR = Path("..")
PROC_DIR = BASE_DIR / "data" / "processed"
DB_PATH  = BASE_DIR / "data" / "raw" / "database.db"

comparisons_df = pd.read_csv(
    PROC_DIR / "comparisons.csv",
    dtype={"winner": str, "loser": str, "intersection_a": str, "intersection_b": str}
)
bt_params_df = pd.read_csv(PROC_DIR / "bt_params.csv", dtype={"intersection_id": str})

# Snelle opzoeking: kruispunt-ID -> ILSR log-sterkte
score_lookup = dict(zip(bt_params_df["intersection_id"], bt_params_df["bt_param_ilsr"]))

print(f"Vergelijkingen: {len(comparisons_df)}")
print(f"Kruispunten met BT-score: {len(bt_params_df)}")

Vergelijkingen: 216
Kruispunten met BT-score: 68


## Stap 2 — Goodness-of-fit: deviance

Het BT-model voorspelt de kans dat kruispunt i van j wint als:
`P(i > j) = exp(s_i) / (exp(s_i) + exp(s_j))`

De deviance is -2 x log-likelihood ten opzichte van een verzadigd model (dat elke vergelijking
perfect voorspelt, met log-likelihood = 0). Een lagere deviance per vergelijking betekent
een betere fit.

In [3]:
def bt_log_likelihood(comparisons: pd.DataFrame, scores: dict) -> float:
    # Som van log P(winner > loser) over alle vergelijkingen
    ll = 0.0
    for _, row in comparisons.iterrows():
        s_w = scores.get(row["winner"], 0.0)
        s_l = scores.get(row["loser"],  0.0)
        # log P(w > l) = s_w - log(exp(s_w) + exp(s_l))  [numeriek stabiel via logaddexp]
        ll += s_w - np.logaddexp(s_w, s_l)
    return ll

ll_model = bt_log_likelihood(comparisons_df, score_lookup)
deviance = -2 * ll_model

print(f"Log-likelihood (BT model):  {ll_model:.2f}")
print(f"Deviance (-2 x LL):         {deviance:.2f}")
print(f"Deviance per vergelijking:  {deviance / len(comparisons_df):.4f}")

Log-likelihood (BT model):  -65.45
Deviance (-2 x LL):         130.89
Deviance per vergelijking:  0.6060


## Stap 3 — Circulaire triaden per respondent

Een circulaire triade is een reeks van drie kruispunten waarbij de keuzes in een kring gaan:
A > B, B > C, maar C > A. Dat is een intransitieve keuze die het BT-model per definitie
niet kan verklaren. Een hoog aantal triaden per respondent wijst op inconsistente beoordelingen.

We itereren over alle combinaties van drie kruispunten die een respondent beoordeeld heeft.

In [4]:
def count_circular_triads(wins: dict) -> int:
    # wins: dict kruispunt-ID -> set van kruispunten die het versloeg
    nodes = list(wins.keys())
    count = 0
    for a, b, c in itertools.combinations(nodes, 3):
        # Beide richtingen van de driehoek controleren
        if (b in wins.get(a, set()) and c in wins.get(b, set()) and a in wins.get(c, set())):
            count += 1
        if (c in wins.get(a, set()) and a in wins.get(b, set()) and b in wins.get(c, set())):
            count += 1
    return count

respondent_quality = []

for r_id, group in comparisons_df.groupby("respondent_id"):
    wins = {}
    for _, row in group.iterrows():
        wins.setdefault(row["winner"], set()).add(row["loser"])

    n_triads = count_circular_triads(wins)
    respondent_quality.append({
        "respondent_id":     r_id,
        "n_comparisons":     len(group),
        "n_circular_triads": n_triads,
    })

quality_df = pd.DataFrame(respondent_quality)
print(quality_df[["n_comparisons", "n_circular_triads"]].describe())

       n_comparisons  n_circular_triads
count            6.0                6.0
mean            36.0                0.0
std              0.0                0.0
min             36.0                0.0
25%             36.0                0.0
50%             36.0                0.0
75%             36.0                0.0
max             36.0                0.0


## Stap 4 — Kendall's W: inter-beoordelaar overeenstemming

Kendall's W meet in hoeverre de respondenten het eens zijn over de rangorde van kruispunten.
We leiden per respondent een impliciete rangvector af op basis van hoeveel wins elk kruispunt
scoorde in zijn vergelijkingen. Vervolgens berekenen we Kendall's W over die vectors.

W = 0 betekent geen overeenstemming, W = 1 betekent perfecte overeenstemming.

In [5]:
all_intersections = sorted(
    set(comparisons_df["intersection_a"]).union(comparisons_df["intersection_b"])
)
n_items   = len(all_intersections)
id_to_pos = {iid: i for i, iid in enumerate(all_intersections)}

# Bouw per-respondent win-count vector (NaN voor kruispunten die ze niet zagen)
rank_matrix_rows = []
for r_id, group in comparisons_df.groupby("respondent_id"):
    win_counts = group["winner"].value_counts()
    row = np.full(n_items, np.nan)
    for iid, cnt in win_counts.items():
        row[id_to_pos[iid]] = cnt
    rank_matrix_rows.append(row)

rank_matrix = np.array(rank_matrix_rows)  # vorm: (n_respondenten, n_kruispunten)

# Bereken W alleen over kruispunten die alle respondenten hebben gezien
observed_cols = ~np.isnan(rank_matrix).any(axis=0)
sub_matrix    = rank_matrix[:, observed_cols]

if sub_matrix.shape[1] > 1:
    ranked   = np.apply_along_axis(rankdata, 1, sub_matrix)
    m, n_sub = ranked.shape
    R        = ranked.sum(axis=0)
    R_mean   = R.mean()
    S        = ((R - R_mean) ** 2).sum()
    W        = 12 * S / (m ** 2 * (n_sub ** 3 - n_sub))
    print(f"Kendall's W (over {n_sub} gezamenlijk geziene kruispunten, {m} respondenten): {W:.4f}")
    print("  0 = geen overeenstemming, 1 = perfecte overeenstemming")
else:
    W = np.nan
    print("Te weinig gezamenlijk geziene kruispunten om Kendall's W te berekenen.")

Kendall's W (over 5 gezamenlijk geziene kruispunten, 6 respondenten): 0.7750
  0 = geen overeenstemming, 1 = perfecte overeenstemming


## Stap 5 — Responstijd-analyse

Bereken voor elke respondent hoe lang ze per taak deden (in seconden).
Taak 1 wordt gemeten vanaf het verlaten van de instructiepagina; elke volgende taak
is het interval tussen twee opeenvolgende knopdrukken.

De analyse is puur informatief — er worden geen respondenten uitgesloten.

In [6]:
con = sqlite3.connect(DB_PATH)
timestamp_df = pd.read_sql("SELECT * FROM Timestamp", con)
con.close()

print(f"Timestamp rijen: {len(timestamp_df)}")
print("Kolommen:", timestamp_df.columns.tolist())
timestamp_df.head(3)

Timestamp rijen: 76
Kolommen: ['respondent_id', 'consent_button', 'screening_button', 'instructions_button', 'act1_task_1_button', 'act1_task_2_button', 'act1_task_3_button', 'act1_task_4_button', 'act1_task_5_button', 'act1_task_6_button', 'act1_task_7_button', 'act1_task_8_button', 'act1_task_9_button', 'act1_task_10_button', 'act1_task_11_button', 'act1_task_12_button', 'act1_task_13_button', 'act1_task_14_button', 'act1_task_15_button', 'act1_task_16_button', 'act1_task_17_button', 'act1_task_18_button', 'act1_task_19_button', 'act1_task_20_button', 'act1_task_21_button', 'act1_task_22_button', 'act1_task_23_button', 'act1_task_24_button', 'act1_task_25_button', 'act1_task_26_button', 'act1_task_27_button', 'act1_task_28_button', 'act1_task_29_button', 'act1_task_30_button', 'act1_task_31_button', 'act1_task_32_button', 'act1_task_33_button', 'act1_task_34_button', 'act1_task_35_button', 'act1_task_36_button', 'act1_task_37_button', 'act1_task_38_button', 'act1_task_39_button', 'ac

,respondent_id,consent_button,screening_button,instructions_button,act1_task_1_button,act1_task_2_button,act1_task_3_button,act1_task_4_button,act1_task_5_button,act1_task_6_button,...,act1_task_292_button,act1_task_293_button,act1_task_294_button,act1_task_295_button,act1_task_296_button,act1_task_297_button,act1_task_298_button,act1_task_299_button,act1_task_300_button,act2_button
0,22697827656,1.777478e+09,1.777478e+09,1.777478e+09,1.777478e+09,1.777478e+09,1.777478e+09,1.777478e+09,NaN,NaN,...,None,None,None,None,None,None,None,None,None,NaN
1,22698105398,1.777479e+09,1.777479e+09,1.777479e+09,NaN,NaN,NaN,NaN,NaN,NaN,...,None,None,None,None,None,None,None,None,None,NaN
2,22698173370,1.777479e+09,1.777479e+09,1.777479e+09,1.777479e+09,1.777479e+09,1.777479e+09,1.777479e+09,1.777479e+09,NaN,...,None,None,None,None,None,None,None,None,None,NaN


In [ ]:
# Filter Timestamp naar alleen de complete respondenten
complete_ids = set(quality_df["respondent_id"])
ts_complete = timestamp_df[timestamp_df["respondent_id"].isin(complete_ids)].copy()

# Bereken responstijd per taak per respondent
# Taak 1: tijd vanaf instructiespagina; taak N: interval tussen opeenvolgende knopdrukken
rt_records = []
for _, row in ts_complete.iterrows():
    r_id = row["respondent_id"]
    prev_ts = row["instructions_button"]
    for pos in range(1, 37):
        curr_ts = row[f"act1_task_{pos}_button"]
        if pd.notna(curr_ts) and pd.notna(prev_ts):
            rt_records.append({
                "respondent_id":  r_id,
                "task_position":  pos,
                "response_time_s": float(curr_ts) - float(prev_ts),
            })
        prev_ts = curr_ts

rt_df = pd.DataFrame(rt_records)

# Aggregeer naar per-respondent statistieken
rt_stats = (
    rt_df.groupby("respondent_id")["response_time_s"]
    .agg(median_rt_s="median", min_rt_s="min", max_rt_s="max")
    .reset_index()
)

quality_df = quality_df.merge(rt_stats, on="respondent_id", how="left")

print("Responstijden per respondent (seconden):")
print(quality_df[["respondent_id", "median_rt_s", "min_rt_s", "max_rt_s"]].to_string(index=False))
print(f"\nOverall mediaan:         {rt_df['response_time_s'].median():.1f}s")
print(f"Snelste individuele taak: {rt_df['response_time_s'].min():.1f}s")
print(f"Traagste individuele taak: {rt_df['response_time_s'].max():.1f}s")

In [ ]:
# Visualiseer de verdeling van responstijden
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram over alle taken en respondenten
axes[0].hist(rt_df["response_time_s"], bins=30, edgecolor="black")
axes[0].set_xlabel("Responstijd (seconden)")
axes[0].set_ylabel("Aantal taken")
axes[0].set_title("Verdeling responstijden (alle taken, alle respondenten)")

# Boxplot per respondent om uitschieters en spreiding te vergelijken
groups = [
    rt_df[rt_df["respondent_id"] == r_id]["response_time_s"].values
    for r_id in quality_df["respondent_id"]
]
axes[1].boxplot(groups, labels=[str(r)[-6:] for r in quality_df["respondent_id"]], vert=True)
axes[1].set_xlabel("Respondent (laatste 6 cijfers ID)")
axes[1].set_ylabel("Responstijd (seconden)")
axes[1].set_title("Responstijden per respondent")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Stap 6 — Opslaan

Sla de per-respondent kwaliteitsmetrics op. Dit bestand kan later gebruikt worden om
twijfelachtige respondenten uit te sluiten en opnieuw te schatten met NB03.

In [8]:
out_path = PROC_DIR / "respondent_quality.csv"
quality_df.to_csv(out_path, index=False)
print(f"Opgeslagen -> {out_path.resolve()}")
quality_df.head()

Opgeslagen -> C:\Users\Thijs\OneDrive\Documents\Studie\EPA\Second_year\Afstuderen\Project\bradley_terry\data\processed\respondent_quality.csv


,respondent_id,n_comparisons,n_circular_triads
0,565642399,36,0
1,851656988,36,0
2,935789013,36,0
3,940909515,36,0
4,1069040705,36,0
